# Visualización Profesional y Síntesis Visual



## Objetivo de la Etapa

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats as scipy_stats
from IPython.display import Image, display

from src.config import PATHS, VIZ
from src.viz import set_style, save_fig, annotate_source
from src.territory import (
 compute_departmental_indicators,
 compute_regional_indicators,
 REGION_COLORS,
 REGION_MAP
)

set_style()
print(f" Ruta de figuras: {PATHS['figures']}")

## 1. Carga de Microdatos Analíticos Integrados

In [ ]:
df_h = pd.read_parquet(PATHS["analytical"] / "df_hogar_analytic.parquet")
df_p = pd.read_parquet(PATHS["analytical"] / "df_persona_analytic.parquet")
df_dpto = compute_departmental_indicators(df_h, df_p)
df_reg = compute_regional_indicators(df_h, df_p)

print(f"Hogares analíticos: {df_h.shape[0]:,} registros (Representan {df_h['fex_c'].sum():,.0f} hogares)")
print(f"Personas analíticas: {df_p.shape[0]:,} registros (Representan {df_p['fex_c'].sum():,.0f} personas)")
print(f"Departamentos: {df_dpto.shape[0]} coberturas geográficas")
print(f"Macro-regiones: {df_reg.shape[0]} zonas territoriales DANE")

## 2. Catálogo Oficial de Visualizaciones del Proyecto

## 3. Figura 17: Panel Ejecutivo de Síntesis (2×2)

In [ ]:
fig = plt.figure(figsize=(16, 11))
fig.suptitle(
 "Radiografía Socioeconómica y Brecha Digital de Colombia\n"
 "Síntesis Visual de Hallazgos Principales — DANE Encuesta de Calidad de Vida (ECV) 2024",
 fontsize=15, weight="bold", y=0.98
)

gs = gridspec.GridSpec(2, 2, figure=fig, wspace=0.32, hspace=0.45)
PALETTE_DIFICULTAD = ["#1a6faf", "#3a96c9", "#f39c12", "#e67e22", "#c0392b"]

# ── PANEL A: Computador por Situación Económica ──────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])

eco_agg = (
 df_h.groupby("sit_eco_escala", as_index=False)
 .apply(lambda g: pd.Series({"pct_comp": (g["tiene_computador"] * g["fex_c"]).sum() / g["fex_c"].sum() * 100}), include_groups=False)
 .sort_values("sit_eco_escala")
)

labels_eco = ["Muy fácil", "Fácil", "Regular", "Difícil", "Muy difícil"]
vals_a = eco_agg["pct_comp"].values

bars_a = ax_a.bar(range(len(vals_a)), vals_a,
 color=PALETTE_DIFICULTAD, edgecolor="white", width=0.62)
for bar, val in zip(bars_a, vals_a):
 ax_a.text(bar.get_x() + bar.get_width()/2, val + 1.5, f"{val:.1f}%",
 ha="center", va="bottom", fontsize=8.5, weight="bold")

nat_comp = (df_h["tiene_computador"] * df_h["fex_c"]).sum() / df_h["fex_c"].sum() * 100
ax_a.axhline(nat_comp, color="#555555", linestyle="--", linewidth=1.2, label=f"Nacional: {nat_comp:.1f}%")

ax_a.set_xticks(range(len(labels_eco)))
ax_a.set_xticklabels(labels_eco, fontsize=9)
ax_a.yaxis.set_major_formatter(mtick.PercentFormatter())
ax_a.set_ylim(0, 92)
ax_a.set_title("A) Tenencia de Computador según Situación Económica del Hogar", fontsize=10.5, weight="bold", pad=8)
ax_a.set_ylabel("% Hogares con Computador", fontsize=9)
ax_a.legend(fontsize=8.5, loc="upper right")

delta = vals_a[0] - vals_a[-1]
ax_a.annotate(f"Brecha: Δ = {delta:.1f} pp",
 xy=(len(vals_a)-1, vals_a[-1]),
 xytext=(len(vals_a)-2.4, vals_a[-1] + 16),
 fontsize=9, color="#c0392b", weight="bold",
 arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.2))

# ── PANEL B: Top-5 y Bottom-5 Departamentos ──────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])

top5 = df_dpto.head(5).sort_values("pct_computador", ascending=True)
bot5 = df_dpto.tail(5).sort_values("pct_computador", ascending=True)

names = list(bot5["departamento"]) + ["—"] + list(top5["departamento"])
vals = list(bot5["pct_computador"]) + [np.nan] + list(top5["pct_computador"])
cols = (["#c0392b"] * 5) + ["white"] + (["#1a6faf"] * 5)

y_pos = range(len(names))
for i, (name, val, col) in enumerate(zip(names, vals, cols)):
 if name == "—":
 ax_b.axhline(i, color="#dddddd", linestyle=":", linewidth=1.0)
 continue
 ax_b.barh(i, val, color=col, alpha=0.85, height=0.62, edgecolor="white")
 ax_b.text(val + 0.8, i, f"{val:.1f}%", va="center", fontsize=8.5, weight="bold")

ax_b.axvline(nat_comp, color="#555555", linestyle="--", linewidth=1.2, label=f"Nacional: {nat_comp:.1f}%")
ax_b.set_yticks(range(len(names)))
ax_b.set_yticklabels(names, fontsize=8.5)
ax_b.xaxis.set_major_formatter(mtick.PercentFormatter())
ax_b.set_xlim(0, 84)
ax_b.set_title("B) Extremos Territoriales: Top-5 y Bottom-5 Departamentos", fontsize=10.5, weight="bold", pad=8)
ax_b.set_xlabel("% Hogares con Computador", fontsize=9)
legend_b = [mpatches.Patch(color="#1a6faf", label="Top-5 Departamentos"),
 mpatches.Patch(color="#c0392b", label="Bottom-5 Departamentos")]
ax_b.legend(handles=legend_b, fontsize=8, loc="lower right")

# ── PANEL C: Fijo vs. Móvil por Macro-Región ─────────────────────────────────
ax_c = fig.add_subplot(gs[1, 0])

reg_order = df_reg.sort_values("pct_internet_movil", ascending=False)["region"].tolist()
x_pos = np.arange(len(reg_order))
width = 0.36

df_reg_sorted = df_reg.set_index("region").loc[reg_order]
bars_fijo = ax_c.bar(x_pos - width/2, df_reg_sorted["pct_internet_fijo"],
 width=width, color="#8c564b", label="Banda Ancha Fija", edgecolor="white")
bars_movil = ax_c.bar(x_pos + width/2, df_reg_sorted["pct_internet_movil"],
 width=width, color="#17becf", label="Internet Móvil", edgecolor="white")

for bar in bars_fijo:
 h = bar.get_height()
 ax_c.text(bar.get_x() + bar.get_width()/2, h + 1.2, f"{h:.0f}%",
 ha="center", va="bottom", fontsize=7.5, color="#8c564b", weight="bold")
for bar in bars_movil:
 h = bar.get_height()
 ax_c.text(bar.get_x() + bar.get_width()/2, h + 1.2, f"{h:.0f}%",
 ha="center", va="bottom", fontsize=7.5, color="#0d7a8a", weight="bold")

ax_c.set_xticks(x_pos)
ax_c.set_xticklabels(reg_order, fontsize=8.5, rotation=10, ha="right")
ax_c.yaxis.set_major_formatter(mtick.PercentFormatter())
ax_c.set_ylim(0, 114)
ax_c.set_title("C) La Paradoja de Conectividad: Fijo vs. Móvil por Región", fontsize=10.5, weight="bold", pad=8)
ax_c.set_ylabel("% Hogares con Conexión", fontsize=9)
ax_c.legend(fontsize=8.5, loc="upper right")

# ── PANEL D: Capital Humano vs. Uso de Internet ──────────────────────────────
ax_d = fig.add_subplot(gs[1, 1])

x = df_dpto["pct_educ_superior_persona"]
y = df_dpto["pct_uso_internet_persona"]
sizes = df_dpto["hogares_expandidos"] / 4500
bubble_colors_d = [REGION_COLORS.get(r, "#555555") for r in df_dpto["region"]]

ax_d.scatter(x, y, s=sizes, c=bubble_colors_d, alpha=0.75, edgecolors="#222222", linewidth=0.6, zorder=3)

slope, intercept, r_val, p_val, _ = scipy_stats.linregress(x, y)
x_line = np.linspace(x.min() * 0.9, x.max() * 1.05, 80)
ax_d.plot(x_line, intercept + slope * x_line, color="#333333", linestyle="--", linewidth=1.5,
 label=f"Ajuste Lineal (R²={r_val**2:.3f})")

label_dptos = {"Bogotá, D.C.": (0.5, -1.2), "Vaupés": (0.5, -3.2), "Vichada": (0.5, -2.8), "Casanare": (-4.2, 1.5), "Antioquia": (0.5, 1.2)}
for _, row in df_dpto.iterrows():
 if row["departamento"] in label_dptos:
 dx, dy = label_dptos[row["departamento"]]
 ax_d.annotate(row["departamento"],
 xy=(row["pct_educ_superior_persona"], row["pct_uso_internet_persona"]),
 xytext=(row["pct_educ_superior_persona"] + dx, row["pct_uso_internet_persona"] + dy),
 fontsize=7.5, weight="bold",
 arrowprops=dict(arrowstyle="->", color="#555555", lw=0.7, shrinkA=2, shrinkB=2))

ax_d.xaxis.set_major_formatter(mtick.PercentFormatter())
ax_d.yaxis.set_major_formatter(mtick.PercentFormatter())
ax_d.set_title(f"D) Capital Humano vs. Uso Individual de Internet (R²={r_val**2:.3f}, p<0.001)", fontsize=10.5, weight="bold", pad=8)
ax_d.set_xlabel("% Personas con Educación Superior", fontsize=9)
ax_d.set_ylabel("% Personas que Usan Internet", fontsize=9)

legend_d = [mpatches.Patch(facecolor=c, label=r) for r, c in REGION_COLORS.items()]
ax_d.legend(handles=legend_d, fontsize=7.5, loc="lower right", framealpha=0.9, title="Región")

annotate_source(fig,
 "Fuente: DANE — Encuesta de Calidad de Vida (ECV) 2024. Estimaciones ponderadas con FEX_C. Cálculos propios.",
 x=0.03, y=-0.015)

save_fig(fig, "fig17_panel_sintesis")
plt.show()

## 4. Figura 18: Forest Plot Comparativo de Determinantes Multivariados

In [ ]:
hogar_model_data = [
 {"label": "Situación eco.: Difícil", "OR": 0.651, "ci_lo": 0.618, "ci_hi": 0.686, "modelo": "Computador en Hogar"},
 {"label": "Situación eco.: Muy difícil", "OR": 0.405, "ci_lo": 0.381, "ci_hi": 0.430, "modelo": "Computador en Hogar"},
 {"label": "Zona: Rural (vs. Urbana)", "OR": 0.413, "ci_lo": 0.392, "ci_hi": 0.434, "modelo": "Computador en Hogar"},
 {"label": "Jefe: Educ. Media", "OR": 2.134, "ci_lo": 2.021, "ci_hi": 2.253, "modelo": "Computador en Hogar"},
 {"label": "Jefe: Educ. Superior", "OR": 6.710, "ci_lo": 6.250, "ci_hi": 7.200, "modelo": "Computador en Hogar"},
 {"label": "Prob. Pobreza Multid. alta", "OR": 0.682, "ci_lo": 0.651, "ci_hi": 0.715, "modelo": "Computador en Hogar"},
]

persona_model_data = [
 {"label": "Alta Dificultad Económica", "OR": 0.748, "ci_lo": 0.718, "ci_hi": 0.779, "modelo": "Uso Internet (Persona)"},
 {"label": "Zona: Rural (vs. Urbana)", "OR": 0.501, "ci_lo": 0.485, "ci_hi": 0.518, "modelo": "Uso Internet (Persona)"},
 {"label": "Educ.: Media (vs. Ninguna)", "OR": 5.820, "ci_lo": 5.350, "ci_hi": 6.330, "modelo": "Uso Internet (Persona)"},
 {"label": "Educ.: Superior (vs. Ninguna)", "OR": 16.92, "ci_lo": 15.50, "ci_hi": 18.47, "modelo": "Uso Internet (Persona)"},
 {"label": "Tiene Celular Propio", "OR": 4.280, "ci_lo": 4.100, "ci_hi": 4.467, "modelo": "Uso Internet (Persona)"},
]

all_or = hogar_model_data + persona_model_data
df_or = pd.DataFrame(all_or).sort_values("OR", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11.5, 8))
model_palette = {"Computador en Hogar": "#1f77b4", "Uso Internet (Persona)": "#2ca02c"}

for i, row in df_or.iterrows():
 color = model_palette[row["modelo"]]
 ax.errorbar(row["OR"], i,
 xerr=[[row["OR"] - row["ci_lo"]], [row["ci_hi"] - row["OR"]]],
 fmt="o", color=color, ecolor=color,
 elinewidth=2.2, capsize=5, capthick=2,
 markersize=9, markeredgecolor="white", markeredgewidth=1.2, zorder=3)

ax.axvline(1.0, color="#cc0000", linestyle="--", linewidth=1.5, zorder=2, label="OR = 1.0 (Efecto Nulo)")
ax.set_xscale("log")
ax.set_yticks(range(len(df_or)))
ax.set_yticklabels(df_or["label"], fontsize=9.5)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda val, _: f"{val:.2g}"))
ax.set_xlabel("Odds Ratio (Escala Logarítmica) con IC del 95%", fontsize=10, labelpad=12)
ax.set_title(
 "Determinantes de la Brecha Digital en Colombia — Forest Plot Comparativo\n"
 "Modelos de Regresión Logística Multivariada (Hogar y Persona) — DANE ECV 2024",
 fontsize=12, weight="bold", pad=15
)

legend_handles = [mpatches.Patch(color=c, label=m) for m, c in model_palette.items()]
legend_handles.append(plt.Line2D([0], [0], color="#cc0000", linestyle="--", label="OR = 1.0 (Nulo)"))
ax.legend(handles=legend_handles, loc="lower right", framealpha=0.95, fontsize=9)

for i, row in df_or.iterrows():
 x_text = row["ci_hi"] * 1.08
 ax.text(x_text, i, f"OR={row['OR']:.2f}", va="center", fontsize=8.5,
 color=model_palette[row["modelo"]], weight="bold")

ax.set_xlim(0.28, 35)
plt.tight_layout()
annotate_source(fig,
 "Fuente: DANE — ECV 2024. Regresión logística multivariada con controles socioeconómicos y geográficos.",
 x=0.04, y=-0.02)

save_fig(fig, "fig18_forest_plot_determinantes")
plt.show()

## 5. Figura 19: Gráfico Dumbbell — Brecha Territorial entre Pobreza y Acceso

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7.5))
fig.suptitle(
 "Exclusión Digital vs. Pobreza Multidimensional — Perfil Territorial de Colombia\n"
 "Comparación en los Departamentos Extremos (ECV 2024, Estimaciones Ponderadas FEX_C)",
 fontsize=13, weight="bold", y=0.98
)

nat_comp = (df_h["tiene_computador"] * df_h["fex_c"]).sum() / df_h["fex_c"].sum() * 100

for ax_idx, (title_label, df_sel) in enumerate([
 ("Top 10 — Mayor Acceso a Computador", df_dpto.head(10).sort_values("pct_computador", ascending=True)),
 ("Bottom 10 — Menor Acceso a Computador", df_dpto.tail(10).sort_values("pct_computador", ascending=True))
]):
 ax = axes[ax_idx]
 df_sel = df_sel.reset_index(drop=True)
 y = range(len(df_sel))

 color_comp = "#1a6faf" if ax_idx == 0 else "#c0392b"
 color_pob = "#e67e22"

 for i, row in df_sel.iterrows():
 comp_val = row["pct_computador"]
 pob_val = row["pct_pobreza_multidimensional"]
 
 # Conexión Dumbbell
 ax.plot([min(comp_val, pob_val), max(comp_val, pob_val)], [i, i],
 color="#bbbbbb", linewidth=2.2, zorder=2)
 
 # Puntos
 ax.plot(comp_val, i, "o", color=color_comp, markersize=10.5, zorder=4)
 ax.plot(pob_val, i, "D", color=color_pob, markersize=8.5, zorder=5)
 
 # Anotaciones
 ax.text(comp_val + 1.2, i, f"{comp_val:.1f}%", va="center", fontsize=8.5, weight="bold", color=color_comp)

 ax.axvline(nat_comp, color="#555555", linestyle="--", linewidth=1.2, zorder=1)

 ax.set_yticks(range(len(df_sel)))
 ax.set_yticklabels(df_sel["departamento"], fontsize=9.5)
 ax.xaxis.set_major_formatter(mtick.PercentFormatter())
 ax.set_xlim(0, 85)
 ax.set_title(title_label, fontsize=11, weight="bold", pad=12)
 ax.set_xlabel("Porcentaje de Hogares (%)", fontsize=9.5, labelpad=10)

# Leyenda compartida superior
legend_elements = [
 plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#1a6faf", markersize=9, label="% Computador (Top 10)"),
 plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#c0392b", markersize=9, label="% Computador (Bottom 10)"),
 plt.Line2D([0], [0], marker="D", color="w", markerfacecolor="#e67e22", markersize=8, label="% Alta Pobreza Multidimensional"),
 plt.Line2D([0], [0], color="#bbbbbb", linewidth=2.2, label="Brecha entre Pobreza y Acceso"),
 plt.Line2D([0], [0], color="#555555", linestyle="--", label=f"Promedio Nacional ({nat_comp:.1f}%)")
]
fig.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, 0.93),
 ncol=5, framealpha=0.95, fontsize=8.5)

plt.tight_layout(rect=[0, 0.03, 1, 0.89])
annotate_source(fig,
 "Fuente: DANE — ECV 2024. Ponderación con FEX_C. Pobreza multidimensional alta: Prob. IAMG ≥ 0.5. Cálculos propios.",
 x=0.03, y=0.01)

save_fig(fig, "fig19_lollipop_brecha_pobreza")
plt.show()

## 6. Conclusiones y Criterios Editoriales de Visualización